# SignBridge — Train the ASL Fingerspelling Gesture Recognizer

Fulfills **PRD F-2 / ML-1** (`docs/PRD.md`) and **tracker task 4.1**
(`docs/MIGRATION_TRACKER.md`): replace the geometric heuristic in
`src/lib/aslClassifier.ts` with a trained MediaPipe Model Maker gesture
recognizer.

**Before you start:**
1. `Runtime` → `Change runtime type` → **T4 GPU** (top menu bar of this Colab).
2. Get a Kaggle API token: kaggle.com → your profile → **Settings** → **API**
   → **Create New Token**. This downloads a `kaggle.json` file you'll upload
   in step 1 below.
3. Expect **~20–30 minutes** end-to-end with the default settings in this
   notebook (600 images/class). Raise `MAX_PER_CLASS_TRAIN` later once the
   pipeline works, per the accuracy floors in PRD F-2 / ML-3.

**What you'll walk away with:**
- `asl-fingerspelling-v1.task` — the trained model, ready to drop into
  `public/models/` (see the last cell for exact next steps).
- `confusion_matrix.png` + `per_class_accuracy.csv` — measured on a held-out
  set the model never saw during training, satisfying the confusion-matrix
  requirement in PRD F-2 / DOC-2.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

ModuleNotFoundError: No module named 'distutils'

## 1. Install dependencies + upload your Kaggle token

In [ ]:
!pip install -q kaggle mediapipe-model-maker scikit-learn matplotlib numpy
print("✓ Dependencies installed")

In [ ]:
from google.colab import files
import os

print("Upload the kaggle.json you downloaded from kaggle.com -> Settings -> API -> Create New Token")
uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError("ERROR: kaggle.json not found. Make sure you uploaded the correct file from Kaggle API settings.")

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✓ Kaggle token installed.")

## 2. Download the ASL Alphabet dataset

[`grassknoted/asl-alphabet`](https://www.kaggle.com/datasets/grassknoted/asl-alphabet)
— 87,000 images, 29 classes (A–Z + SPACE/DELETE/NOTHING), ~1 GB zipped.

In [ ]:
import os
import subprocess
import time

max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"Downloading dataset (attempt {attempt + 1}/{max_retries})...")
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', 'grassknoted/asl-alphabet', '-p', '/content/data', '--unzip'],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            if os.path.exists('/content/data') and len(os.listdir('/content/data')) > 0:
                print("✓ Dataset downloaded successfully")
                break
        else:
            print(f"Download failed: {result.stderr}")
    except Exception as e:
        print(f"Error: {e}")
    
    if attempt < max_retries - 1:
        print(f"Retrying in 10 seconds...")
        time.sleep(10)
else:
    raise RuntimeError("Failed to download dataset after 3 attempts. Check Kaggle auth or internet connection.")

## 3. Filter to SignBridge's supported classes

Two things happen here:

- **J and Z are excluded.** They're motion gestures — unrecognizable from a
  single frame — and are tracked as their own follow-up (PRD **F-7** / **ML-4**,
  a temporal sequence model). Training them here as if they were static
  letters would just teach the model a wrong shape for both.
- **One folder must be named `none`** — MediaPipe Model Maker's gesture
  recognizer *requires* exactly one such class for "no matching gesture."
  The dataset's own `nothing` folder (empty frame, no hand) maps onto this
  directly.

We also carve out a **held-out set that Model Maker never sees** during
training — `HOLDOUT_ROOT` below — so the confusion matrix in step 6 measures
real generalization, not memorized training images.

In [ ]:
import glob, os, shutil, random

random.seed(42)

candidates = glob.glob('/content/data/**/A', recursive=True)
if not candidates:
    candidates = glob.glob('/content/data/asl-alphabet/train/A', recursive=False)
if not candidates:
    print("DEBUG: Directory structure:")
    for root, dirs, files in os.walk('/content/data'):
        level = root.replace('/content/data', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        if level < 3:
            for dir_name in dirs[:5]:
                print(f'{indent}  {dir_name}/')
    raise RuntimeError(
        "Could not find 'A' class folder in expected location. "
        "Check the dataset structure with the DEBUG output above."
    )

SRC_ROOT = os.path.dirname(candidates[0])
print(f"✓ Detected training folder: {SRC_ROOT}")

TRAIN_POOL_ROOT = '/content/asl_train_filtered'
HOLDOUT_ROOT = '/content/asl_holdout'
os.makedirs(TRAIN_POOL_ROOT, exist_ok=True)
os.makedirs(HOLDOUT_ROOT, exist_ok=True)

STATIC_LETTERS = [chr(c) for c in range(ord('A'), ord('Z') + 1) if chr(c) not in ('J', 'Z')]

MAX_PER_CLASS_TRAIN = 600
HOLDOUT_PER_CLASS = 60

def split_copy(class_name, src_dir):
    if not os.path.exists(src_dir):
        raise FileNotFoundError(f"Source folder missing: {src_dir}")
    
    dst_train = os.path.join(TRAIN_POOL_ROOT, class_name)
    dst_holdout = os.path.join(HOLDOUT_ROOT, class_name)
    os.makedirs(dst_train, exist_ok=True)
    os.makedirs(dst_holdout, exist_ok=True)
    
    imgs = [f for f in os.listdir(src_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if len(imgs) < HOLDOUT_PER_CLASS:
        print(f"WARNING: {class_name} has only {len(imgs)} images, need {HOLDOUT_PER_CLASS} for holdout")
    
    random.shuffle(imgs)
    holdout_imgs = imgs[:HOLDOUT_PER_CLASS]
    train_imgs = imgs[HOLDOUT_PER_CLASS:HOLDOUT_PER_CLASS + MAX_PER_CLASS_TRAIN]
    
    for name in holdout_imgs:
        shutil.copy(os.path.join(src_dir, name), os.path.join(dst_holdout, name))
    for name in train_imgs:
        shutil.copy(os.path.join(src_dir, name), os.path.join(dst_train, name))

for letter in STATIC_LETTERS:
    split_copy(letter, os.path.join(SRC_ROOT, letter))

split_copy('none', os.path.join(SRC_ROOT, 'nothing'))

print("✓ Train-pool classes:", sorted(os.listdir(TRAIN_POOL_ROOT)))
print("✓ Holdout classes:   ", sorted(os.listdir(HOLDOUT_ROOT)))

## 4. Load & split the dataset

Model Maker runs MediaPipe's hand landmark detector over every image and
trains on the extracted landmarks, not raw pixels -- images with no
detectable hand are dropped automatically.

In [ ]:
from mediapipe_model_maker import gesture_recognizer
import os

try:
    print("Loading dataset (this may take 2-5 minutes for 87K images)...")
    data = gesture_recognizer.Dataset.from_folder(
        dirname=TRAIN_POOL_ROOT,
        hparams=gesture_recognizer.HandDataPreprocessingParams(),
    )
    
    if data.size == 0:
        raise ValueError("Dataset loaded but contains 0 samples. Check image integrity.")
    
    train_data, rest_data = data.split(0.8)
    validation_data, test_data = rest_data.split(0.5)
    
    print(f"✓ Dataset loaded:")
    print(f"  train={train_data.size}  validation={validation_data.size}  test={test_data.size}")
    
    if train_data.size < 100:
        print("WARNING: Very small training set. Model may not train well.")
    
    if validation_data.size == 0 or test_data.size == 0:
        raise ValueError("Validation or test set is empty. Some classes lost all samples during hand landmark extraction.")
    
except Exception as e:
    raise RuntimeError(f"Failed to load dataset: {str(e)} Check for corrupted images in {TRAIN_POOL_ROOT}")

## 5. Train

Defaults below (20 epochs) are a reasonable starting point. If accuracy in
step 6 misses the PRD floors, raise `epochs` and/or `MAX_PER_CLASS_TRAIN`
above and rerun from step 3.

In [ ]:
hparams = gesture_recognizer.HParams(
    export_dir='/content/exported_model',
    epochs=20,
    batch_size=16,
    learning_rate=0.001,
)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options,
)

## 6. Evaluate & export

In [ ]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Overall test loss: {loss:.4f}   Overall test accuracy: {acc:.4f}")

model.export_model()
!ls -la /content/exported_model

## 7. Per-class confusion matrix (PRD F-2 / ML-3 accuracy floors)

This runs the **exported `.task` file** through the same MediaPipe Tasks
GestureRecognizer API `useSignDetector.ts` will use in the app, against the
held-out images from step 3 that training never saw. This is the number that
matters -- not the aggregate `acc` above.

Targets from `docs/PRD.md`:
- Reliable set (A,B,C,D,F,G,H,I,K,L,O,R,U,V,W,X,Y): **>=95%**
- Weak set (E,S,T,M,N,P,Q): **>=85%**, none below 80%

In [ ]:
import os, csv
import mediapipe as mp
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

TASK_PATH = '/content/exported_model/gesture_recognizer.task'

if not os.path.exists(TASK_PATH):
    raise FileNotFoundError(f"Model file not found: {TASK_PATH}. Training may have failed.")

try:
    BaseOptions = mp.tasks.BaseOptions
    GestureRecognizer = mp.tasks.vision.GestureRecognizer
    GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    recognizer = GestureRecognizer.create_from_options(GestureRecognizerOptions(
        base_options=BaseOptions(model_asset_path=TASK_PATH),
        running_mode=VisionRunningMode.IMAGE,
    ))
    
    print("✓ Model loaded successfully")
except Exception as e:
    raise RuntimeError(f"Failed to load gesture recognizer: {str(e)}")

classes = sorted(os.listdir(HOLDOUT_ROOT))
y_true, y_pred = [], []
no_hand_count = 0

try:
    for label in classes:
        folder = os.path.join(HOLDOUT_ROOT, label)
        for fname in os.listdir(folder):
            try:
                image = mp.Image.create_from_file(os.path.join(folder, fname))
                result = recognizer.recognize(image)
                if not result.gestures or not result.gestures[0]:
                    no_hand_count += 1
                    continue
                y_true.append(label)
                y_pred.append(result.gestures[0][0].category_name)
            except Exception as e:
                print(f"Warning: Failed to process {fname}: {e}")
                no_hand_count += 1
                continue
except Exception as e:
    raise RuntimeError(f"Evaluation failed: {str(e)}")

unknown = set(y_pred) - set(classes)
if unknown:
    raise ValueError(f"Model predicted unknown categories: {unknown}. Model file may be corrupted.")

print(f"✓ Holdout images with no detected hand (excluded): {no_hand_count}\n")
print(classification_report(y_true, y_pred, labels=classes, zero_division=0))

try:
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=90)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('SignBridge ASL gesture recognizer -- held-out confusion matrix')
    fig.colorbar(im)
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=150)
    plt.show()

    with open('/content/per_class_accuracy.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['letter', 'accuracy', 'n_holdout'])
        for i, label in enumerate(classes):
            total = cm[i].sum()
            correct = cm[i][i]
            accuracy = (correct / total if total else 0)
            w.writerow([label, f"{accuracy:.4f}", int(total)])
            
            if total == 0 and label == 'none':
                print(f"⚠ WARNING: Class '{label}' has zero predictions; all holdout images lacked detectable hand.")

    print("✓ Saved /content/confusion_matrix.png and /content/per_class_accuracy.csv")
except Exception as e:
    raise RuntimeError(f"Failed to save metrics: {str(e)}")

## 8. Download the artifacts

In [ ]:
from google.colab import files
import os

files_to_download = [
    '/content/exported_model/gesture_recognizer.task',
    '/content/confusion_matrix.png',
    '/content/per_class_accuracy.csv'
]

print("Downloading trained model artifacts...")
for file_path in files_to_download:
    if os.path.exists(file_path):
        try:
            files.download(file_path)
            print(f"✓ Downloaded {os.path.basename(file_path)}")
        except Exception as e:
            print(f"ERROR downloading {file_path}: {e}")
    else:
        print(f"ERROR: File not found {file_path}")

print("\n✓ All artifacts downloaded. Check your Downloads folder.")

## 9. Bring the model back into SignBridge

1. **Check `per_class_accuracy.csv` against the floors** (PRD F-2 / ML-3):
   reliable set >=95%, weak set (E,S,T,M,N,P,Q) >=85%, nothing below 80%. If a
   letter misses, raise `MAX_PER_CLASS_TRAIN` and/or `epochs` above and rerun
   from step 3 -- don't ship a model that fails its own published floor.
2. **Rename and version** the downloaded `gesture_recognizer.task` ->
   `asl-fingerspelling-v1.task`. Never overwrite an existing versioned file
   in place (tracker task 4.4 / OPS-4) -- `/public/models` and `/public/wasm`
   are served with a 1-year immutable cache header (see `vercel.json`), so an
   in-place overwrite would leave existing users stuck on the old model for a
   year.
3. Place it at `public/models/asl-fingerspelling-v1.task` in the repo.
4. Save `confusion_matrix.png` under `docs/models/` and note the version +
   headline accuracy numbers in `docs/SIGN_DETECTION.md`.
5. Ask your coding assistant to do tracker **task 4.3**: swap
   `classifyASL()` for `GestureRecognizer` in `src/hooks/useSignDetector.ts`
   (sibling API to the `HandLandmarker` already loaded there), keeping the
   geometric heuristic as an offline fallback behind a flag, and **task
   4.10**: update the in-app Accuracy popover if the weak-letter set changed.